![The blog’s ninja owl exploring bootstrap intervals](https://howtolearnmachinelearning.com/wp-content/uploads/2026/11/bootstrap-confidence-intervals-anime-owl.png)

# Bootstrap confidence intervals in Python

**Hey friends!** Two models predicted the same 300 cases. The quadratic model seems a little better. Your team will switch only if the evidence supports a reduction of **at least 0.05 units** in mean absolute error (MAE). Would you sign off?

Run each cell in order. Make your picks in the form fields (or edit the quoted values in Jupyter). You can retry any decision. [Read the companion article](https://howtolearnmachinelearning.com/articles/bootstrap-confidence-intervals-python/).

## 1. Meet your two candidates
Each row represents one independent case. Both models train on the same 900 examples. Look at the first five test cases below. What makes a case an especially useful unit for comparing these models?

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from sklearn.linear_model import LinearRegression
from sklearn.model_selection import train_test_split
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import PolynomialFeatures

rng = np.random.default_rng(42)
X = rng.uniform(-2, 2, (1200, 1))
y = 2 * X[:, 0] + 0.5 * X[:, 0] ** 2 + rng.normal(0, 2, 1200)
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.25, random_state=42)
linear = LinearRegression().fit(X_train, y_train)
quadratic = make_pipeline(PolynomialFeatures(2, include_bias=False), LinearRegression()).fit(X_train, y_train)
err_a = np.abs(y_test - linear.predict(X_test))
err_b = np.abs(y_test - quadratic.predict(X_test))
print("Case   Linear absolute error   Quadratic absolute error")
for i in range(5):
    print(f"{i+1:4}   {err_a[i]:21.3f}   {err_b[i]:24.3f}")
delta = (err_b - err_a).mean()
print(f"Observed MAE difference (quadratic minus linear): {delta:+.3f}")

## 2. Design the resampling experiment
Pick a design and predict what will happen to the interval width if you break the pairing. Run this cell, then try the other design. Each experiment draws 300 cases with replacement 5,000 times.

In [ ]:
# @title Run your experiment
design = "Choose" # @param ["Choose", "Same case IDs", "Separate case IDs"]
prediction = "Choose" # @param ["Choose", "Breaking pairs widens it", "Breaking pairs narrows it"]

def bootstrap(draw_together):
    generator = np.random.default_rng(123)
    n = len(err_a)
    rows_a = generator.integers(n, size=(5000, n))
    rows_b = rows_a if draw_together else generator.integers(n, size=(5000, n))
    return err_b[rows_b].mean(axis=1) - err_a[rows_a].mean(axis=1)

if design == "Choose" or prediction == "Choose":
    print("Choose a design and a width prediction, then run again.")
else:
    together = design == "Same case IDs"
    draws = bootstrap(together)
    other = bootstrap(not together)
    low, high = np.quantile(draws, [0.025, 0.975])
    other_width = np.ptp(np.quantile(other, [0.025, 0.975]))
    widened = high - low < other_width if together else high - low > other_width
    correct = widened == (prediction == "Breaking pairs widens it")
    print(f"{design}: 95% percentile interval [{low:+.3f}, {high:+.3f}].")
    print(f"Your interval width: {high-low:.3f}. Other design: {other_width:.3f}.")
    print("Your width prediction fits." if correct else "Revisit your prediction using the two widths.")
    print("Which design keeps a difficult case difficult for BOTH models in the same draw?")
    plt.hist(draws, bins=45, color="#b02762")
    plt.axvline(0, color="black", label="Equal MAE")
    plt.axvline(-0.05, color="#116659", ls="--", label="Required gain")
    plt.xlabel("MAE difference (quadratic minus linear)")
    plt.ylabel("Resamples")
    plt.legend()
    plt.show()

## 3. Make the release call
Use the **paired** interval. The team requires the entire 95% interval to support its minimum improvement. Choose the right design above, then decide below. After your first decision, lower the required gain to 0.02. What changes?

In [ ]:
# @title Your decision
required_gain = 0.05 # @param [0.02, 0.05] {type:"raw"}
decision = "Choose" # @param ["Choose", "Switch models", "Keep the current model"]
if design != "Same case IDs" or prediction == "Choose":
    print("First run Step 2 using the same test cases for both models.")
elif decision == "Choose":
    print("Compare the end of the interval closest to zero with the required gain, then choose.")
else:
    upper = np.quantile(bootstrap(True), 0.975)
    qualifies = upper <= -required_gain
    print(f"Smallest gain supported by the interval: {-upper:.3f}. Required: {required_gain:.3f}.")
    if (decision == "Switch models") == qualifies:
        print("Your decision follows the release rule. Try the other minimum gain.")
    else:
        print("Try again. Would every gain inside the interval clear the team's cutoff?")

## Worked answers

1. **Keep both errors from each case together.** Some test cases are tough for both models. A shared row draw preserves that relationship. Separate draws compare different mixes of easy and difficult cases and give a wider interval in this example.
2. **The quadratic model improves MAE by about 0.090.** The paired 95% percentile interval for quadratic minus linear is about **[−0.146, −0.033]**. Negative differences favor the quadratic model. The plausible gain spans roughly 0.033 to 0.146 units.
3. **Keep the current model at the 0.05 cutoff.** A gain of 0.033 would fall short, even though the point estimate clears the cutoff. **Switch at 0.02** under the stated rule. The evidence stays the same across those two decisions. The cost the team is willing to accept changes.
4. **What uncertainty did we measure?** These intervals describe test-case sampling variation for two fixed fitted models, assuming independent, representative cases. Estimating variation from retraining would require repeating the training process. If multiple rows came from one customer, consider resampling whole customers together.

**One last question for your own project:** What improvement would justify switching models, and who could help you choose that number before you look at test results?